In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from tqdm import tqdm

In [2]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36"
}

In [3]:
def get_html(url):
    resp = requests.get(url, headers=headers, timeout=15)
    resp.raise_for_status()
    return resp.text

In [7]:
def scrape_player_bio(player_id):
    # name, height_cm, weight_kg, position, shoots, college, draft_yearو birth_date, birth_place, experience
 
    first_letter = player_id[0]
    url = "https://www.basketball-reference.com/players/" + first_letter + "/" + player_id + ".html"
 
    html = get_html(url)
    soup = BeautifulSoup(html, "html.parser")
 
    meta_div = soup.find("div", {"id": "meta"})
 
    result = {"player_id": player_id}

    name_tag = meta_div.find("h1")
    if name_tag is not None:
        result["name"] = name_tag.get_text(strip=True)
 
    paragraphs = meta_div.find_all("p")
 
    for p in paragraphs:
        text = p.get_text(" ", strip=True)
 
        # Height and weight
        if "cm" in text and "kg" in text:
            match = re.search(r"\((\d+)cm,\s*(\d+)kg\)", text)
            if match:
                result["height_cm"] = int(match.group(1))
                result["weight_kg"] = int(match.group(2))
 
        # Position and shoot  
        if "Position:" in text:
            after_position = text.split("Position:")[1]
            result["position"] = after_position.split("▪")[0].strip()
 
            if "Shoots:" in text:
                after_shoots = text.split("Shoots:")[1].strip()
                result["shoots"] = after_shoots.split()[0]
 
        # College
        if "College:" in text:
            college_link = p.find("a")
            if college_link is not None:
                result["college"] = college_link.get_text(strip=True)
 
        # Draft
        if "Draft:" in text:
            match = re.search(r"(\d{4}) NBA Draft", text)
            if match:
                result["draft_year"] = int(match.group(1))

        # NBA Debut
        if "NBA Debut:" in text:
            debut_link = p.find("a")
            if debut_link is not None:
                result["nba_debut"] = debut_link.get_text(strip=True)

        # Experince for acitve player
        if "Experience:"in text:
            match = re.search(r"(?:Experience):\s*(\d+)\s*years?", text)
            if match:
                result["experience_years"] = int(match.group(1))
                result["is_active"] = True

        # Career lenght for retierd player
        if "Career Length:"in text:
            match = re.search(r"(?:Career Length):\s*(\d+)\s*years?", text)
            if match:
                result["experience_years"] = int(match.group(1))
                result["is_active"] = False
        
 
    # birth date and birth place same span
    birth_span = meta_div.find("span", {"id": "necro-birth"})
    if birth_span is not None:
        result["birth_date"] = birth_span.get("data-birth")
 
        birth_p = birth_span.find_parent("p")
        links_in_p = birth_p.find_all("a")
        if links_in_p:
            result["birth_place"] = links_in_p[-1].get_text(strip=True)
 
    return result

In [8]:
total_stats_df = pd.read_csv("data\player_total_stats.csv")
unique_ids = total_stats_df['player_url_id'].dropna().unique()
print("count of unique players:", len(unique_ids))

count of unique players: 1172


In [13]:
# test_ids = unique_ids[:5]

bios = []
for pid in tqdm(unique_ids):
    bio = scrape_player_bio(pid)
    bios.append(bio)
    time.sleep(4)
 
bio_df = pd.DataFrame(bios)

100%|██████████| 1172/1172 [1:41:45<00:00,  5.21s/it]  


In [14]:
bio_df.head()

,player_id,name,position,shoots,height_cm,weight_kg,college,draft_year,nba_debut,experience_years,is_active,birth_date,birth_place
0,hardeja01,James Harden,Point Guard and Shooting Guard,Left,196,99,Arizona State,2009.0,"October 28, 2009",17,True,1989-08-26,California
1,lillada01,Damian Lillard,Point Guard,Right,188,90,Weber State,2012.0,"October 31, 2012",13,True,1990-07-15,California
2,bookede01,Devin Booker,Shooting Guard and Point Guard,Right,196,93,Kentucky,2015.0,"October 28, 2015",11,True,1996-10-30,Michigan
3,antetgi01,Giannis Antetokounmpo,"Power Forward, Small Forward, Point Guard, and...",Right,211,110,NaN,2013.0,"October 30, 2013",13,True,1994-12-06,Greece
4,youngtr01,Trae Young,Point Guard,Right,188,74,Oklahoma,2018.0,"October 17, 2018",8,True,1998-09-19,Texas


In [15]:
bio_df.tail()

,player_id,name,position,shoots,height_cm,weight_kg,college,draft_year,nba_debut,experience_years,is_active,birth_date,birth_place
1167,grayha01,Hayden Gray,Shooting Guard,Right,193,86,UC San Diego,NaN,"April 12, 2026",1,True,2003-05-11,California
1168,telfoja01,Jahmyl Telfort,Power Forward,Right,201,102,NaN,NaN,"October 28, 2025",1,True,2001-04-30,Quebec
1169,brownda04,Darius Brown II,Shooting Guard,Right,188,87,NaN,NaN,"February 24, 2026",1,True,1999-07-28,California
1170,essenno01,Noa Essengue,Power Forward,Right,203,90,NaN,2025.0,"November 22, 2025",1,True,2006-12-18,France
1171,hepbuch01,Chucky Hepburn,Point Guard,Right,183,86,NaN,NaN,"November 30, 2025",1,True,2003-02-09,Nebraska


In [16]:
bio_df.to_csv("player_bios.csv", index=False, encoding="utf-8-sig")